In [1]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=False
)

pdf_documents = loader.load()

pdf_documents

C:\Users\krish\AppData\Local\Temp\ipykernel_15412\4268358651.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-03T09:35:44+00:00', 'source': '..\\data\\pdf\\01_ai_intro.pdf', 'file_path': '..\\data\\pdf\\01_ai_intro.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-10-03T09:35:44+00:00', 'trapped': '', 'modDate': "D:20261003093544+00'00'", 'creationDate': "D:20261003093544+00'00'", 'page': 0}, page_content='Introduction to Artificial Intelligence\nWhat is Artificial Intelligence?\nArtificial Intelligence (AI) is the field of computer science concerned with building systems that can\nperform tasks that normally require human-like capabilities, such as perception, reasoning, learning,\nlanguage understanding, and decision making.\nMajor Areas of AI\nImportant areas include machine learning, natural language processing, computer vision, robotics,\nknowledge representation

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(pdf_documents)

print("Number of chunks:", len(chunks))

print("\nFirst chunk:")
print(chunks[0].page_content)

print("\nMetadata:")
print(chunks[0].metadata)

Number of chunks: 12

First chunk:
Introduction to Artificial Intelligence
What is Artificial Intelligence?
Artificial Intelligence (AI) is the field of computer science concerned with building systems that can
perform tasks that normally require human-like capabilities, such as perception, reasoning, learning,
language understanding, and decision making.
Major Areas of AI
Important areas include machine learning, natural language processing, computer vision, robotics,

Metadata:
{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-03T09:35:44+00:00', 'source': '..\\data\\pdf\\01_ai_intro.pdf', 'file_path': '..\\data\\pdf\\01_ai_intro.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-10-03T09:35:44+00:00', 'trapped': '', 'modDate': "D:20261003093544+00'00'", 'creationDate': "D:20261003093544+00'00'", 'page': 0}


In [3]:
chunks


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-03T09:35:44+00:00', 'source': '..\\data\\pdf\\01_ai_intro.pdf', 'file_path': '..\\data\\pdf\\01_ai_intro.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-10-03T09:35:44+00:00', 'trapped': '', 'modDate': "D:20261003093544+00'00'", 'creationDate': "D:20261003093544+00'00'", 'page': 0}, page_content='Introduction to Artificial Intelligence\nWhat is Artificial Intelligence?\nArtificial Intelligence (AI) is the field of computer science concerned with building systems that can\nperform tasks that normally require human-like capabilities, such as perception, reasoning, learning,\nlanguage understanding, and decision making.\nMajor Areas of AI\nImportant areas include machine learning, natural language processing, computer vision, robotics,'),
 Document(metadata={'p

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [5]:
vectors = embeddings.embed_documents(
    [chunk.page_content for chunk in chunks]
)

In [6]:
vectors

[[0.02216554805636406,
  -0.021063974127173424,
  -0.0008271678816527128,
  0.001367804012261331,
  0.010358152911067009,
  9.357443923363462e-05,
  0.08312976360321045,
  0.0483844168484211,
  0.015300342813134193,
  0.022590605542063713,
  -0.0013092802837491035,
  -0.03614594042301178,
  0.005973130464553833,
  0.037514615803956985,
  0.01725921593606472,
  0.0069557540118694305,
  -0.03776712343096733,
  -0.020831992849707603,
  0.01731371507048607,
  -0.028609827160835266,
  0.06406468898057938,
  0.03545781597495079,
  -0.04795731231570244,
  -0.062482379376888275,
  -0.07828964293003082,
  0.032688286155462265,
  -0.009649553336203098,
  -0.04074205085635185,
  -0.04296516254544258,
  -0.05402308329939842,
  0.002571358811110258,
  0.031395383179187775,
  0.09998228400945663,
  -0.011942374520003796,
  0.017324745655059814,
  0.0491936169564724,
  0.028596404939889908,
  0.037818294018507004,
  -0.018166346475481987,
  0.014624197036027908,
  -0.030389275401830673,
  -0.08872202

In [7]:
from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding = embeddings
)

In [ ]:
import os

groq_api_key = os.getenv("GROQ_API_KEY")

: 

In [9]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)


In [10]:
# Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 3,
        "score_threshold": 0.5
    }
)

question = "What is artificial intelligence?"

results = retriever.invoke(question)

if not results:
    print("I don't know based on the provided document.")

else:
    context = "\n\n".join(
        result.page_content for result in results
    )

    prompt = f"""
Answer the question using only the provided context.

If the answer cannot be found in the context, say:
"I don't know based on the provided document."

Context:
{context}

Question:
{question}

Answer:
"""

    response = llm.invoke(prompt)

    print("Answer:")
    print(response.content)

    print("\nSources:")
    for result in results:
        print("PDF:", result.metadata.get("source"))
        print("Page:", result.metadata.get("page"))

Answer:
Artificial Intelligence (AI) is the field of computer science concerned with building systems that can perform tasks that normally require human‑like capabilities, such as perception, reasoning, learning, language understanding, and decision making.

Sources:
PDF: ..\data\pdf\01_ai_intro.pdf
Page: 0
PDF: ..\data\pdf\01_ai_intro.pdf
Page: 0
PDF: ..\data\pdf\01_ai_intro.pdf
Page: 0


In [11]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

def ingest_pdf(pdf_path):
    loader = PyMuPDFLoader(pdf_path)
    documents = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )

    chunks = text_splitter.split_documents(documents)

    vectorstore = Chroma.from_documents(
        collection_name="rag_documents",
        documents=chunks,
        embedding=embeddings,

    )

    return vectorstore

In [12]:
def ask_question(question, vectorstore):
    retriever = vectorstore.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={
            "k": 3,
            "score_threshold": 0.5
        }
    )

    results = retriever.invoke(question)

    if not results:
        return {
            "answer": "I don't know based on the provided document.",
            "sources": []
        }

    context = ""

    for result in results:
        context += result.page_content + "\n\n"

    prompt = f"""
Answer the question using only the context below.

If the answer is not present in the context, say:
"I don't know based on the provided document."

Context:
{context}

Question:
{question}

Answer:
"""

    response = llm.invoke(prompt)

    sources = []

    for result in results:
        sources.append({
            "source": result.metadata.get("source"),
            "page": result.metadata.get("page")
        })

    return {
        "answer": response.content,
        "sources": sources
    }

In [13]:
vectorstore = ingest_pdf("../data/pdf/01_ai_intro.pdf")

result = ask_question(
    "What is artificial intelligence?",
    vectorstore
)

print("Answer:")
print(result["answer"])

print("\nSources:")

for source in result["sources"]:
    print("PDF:", source["source"])
    print("Page:", source["page"])

Answer:
Artificial Intelligence (AI) is the field of computer science concerned with building systems that can perform tasks that normally require human‑like capabilities, such as perception, reasoning, learning, language understanding, and decision making.

Sources:
PDF: ../data/pdf/01_ai_intro.pdf
Page: 0
PDF: ../data/pdf/01_ai_intro.pdf
Page: 0
PDF: ../data/pdf/01_ai_intro.pdf
Page: 0
